# Sleep Data Exploration

Explore Oura sleep data: sleep stages, HRV trends, efficiency, and sleep architecture over time.

**Prerequisites:** Run `pip install -e ".[dev]"` from the project root and configure your `.env` file.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from oura_client import OuraClient

# Toggle sandbox=True to use mock data without a real ring
client = OuraClient(sandbox=True)

# Adjust date range as needed
START = "2025-01-01"
END = "2025-03-01"

## Fetch Sleep Data

In [ ]:
sleep_periods = client.get_sleep(start_date=START, end_date=END)
print(f"Fetched {len(sleep_periods)} sleep periods")

df = pd.DataFrame([
    {
        "day": s.day,
        "total_sleep_h": s.total_sleep_hours,
        "deep_h": s.deep_sleep_hours,
        "light_h": s.light_sleep_hours,
        "rem_h": s.rem_sleep_hours,
        "awake_h": s.awake_hours,
        "efficiency": s.efficiency,
        "avg_hrv": s.average_hrv,
        "avg_hr": s.average_heart_rate,
        "lowest_hr": s.lowest_heart_rate,
        "avg_breath": s.average_breath,
        "latency_min": (s.latency or 0) / 60,
        "restless_periods": s.restless_periods,
        "type": s.type.value,
    }
    for s in sleep_periods
])
df = df.sort_values("day").reset_index(drop=True)
df.head(10)

## Sleep Architecture Over Time

Stacked bar chart showing deep, light, REM, and awake time per night.

In [ ]:
fig = go.Figure()
for stage, color in [("deep_h", "#1a237e"), ("light_h", "#42a5f5"), ("rem_h", "#ab47bc"), ("awake_h", "#ef5350")]:
    fig.add_trace(go.Bar(x=df["day"], y=df[stage], name=stage.replace("_h", "").title(), marker_color=color))
fig.update_layout(
    barmode="stack",
    title="Sleep Architecture",
    xaxis_title="Date",
    yaxis_title="Hours",
    template="plotly_white",
)
fig.show()

## HRV Trend During Sleep

In [ ]:
hrv_df = df.dropna(subset=["avg_hrv"])
fig = px.line(
    hrv_df, x="day", y="avg_hrv",
    title="Average HRV During Sleep",
    labels={"avg_hrv": "HRV (ms)", "day": "Date"},
    template="plotly_white",
)
fig.add_hline(y=hrv_df["avg_hrv"].mean(), line_dash="dash", annotation_text="Mean", line_color="gray")
fig.show()

## Sleep Efficiency Over Time

In [ ]:
eff_df = df.dropna(subset=["efficiency"])
fig = px.scatter(
    eff_df, x="day", y="efficiency",
    size="total_sleep_h", color="latency_min",
    title="Sleep Efficiency (bubble size = total sleep, color = latency)",
    labels={"efficiency": "Efficiency %", "day": "Date", "latency_min": "Latency (min)"},
    template="plotly_white",
)
fig.add_hline(y=85, line_dash="dot", annotation_text="Good threshold", line_color="green")
fig.show()

## Resting Heart Rate Trend

In [ ]:
hr_df = df.dropna(subset=["avg_hr", "lowest_hr"])
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=hr_df["day"], y=hr_df["avg_hr"], name="Avg HR", mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=hr_df["day"], y=hr_df["lowest_hr"], name="Lowest HR", mode="lines+markers"), secondary_y=False)
fig.update_layout(title="Heart Rate During Sleep", template="plotly_white")
fig.update_yaxes(title_text="BPM", secondary_y=False)
fig.show()

## Summary Statistics

In [ ]:
summary_cols = ["total_sleep_h", "deep_h", "rem_h", "efficiency", "avg_hrv", "avg_hr", "latency_min"]
df[summary_cols].describe().round(2)